# Практика 1. NumPy, pandas и точка отсчёта

**Курс «Машинное обучение» · Часть 1, занятие 1**

Сегодня занимаемся инструментами. Без них дальнейший курс невозможен: любая модель начинается с того, что данные лежат в таблице и их нужно достать, отфильтровать и посчитать.

План:

1. **NumPy** — массивы и векторизованные вычисления
2. **pandas** — таблицы: выбор данных, фильтрация, столбцы, группировки
3. **Точка отсчёта** — почему модель нельзя оценивать без baseline

Разведочный анализ, графики и настоящие модели — на следующих занятиях. Сегодня руки привыкают к инструменту.

> Ячейки с пометкой **Задание** нужно заполнить самостоятельно. Сразу после каждого задания идёт ячейка проверки: если она отработала без ошибки, задание сделано верно.

In [59]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

RANDOM_STATE = 42   # фиксируем случайность, чтобы результаты воспроизводились

print("numpy:", np.__version__)
print("pandas:", pd.__version__)

numpy: 2.1.3
pandas: 2.2.3


---
## 1. NumPy: почему не обычные списки

Список в Python хранит указатели на объекты, разбросанные по памяти. NumPy-массив — непрерывный кусок памяти с числами одного типа. Отсюда две вещи: он компактнее и операции над ним выполняются на порядки быстрее.

In [60]:
a = np.array([1, 2, 3, 4, 5])
b = np.array([10, 20, 30, 40, 50])

print("сумма поэлементно:", a + b)
print("умножение поэлементно:", a * b)
print("возведение в степень:", a ** 2)

# Со списками это не сработает: [1,2,3] + [10,20,30] просто склеит их
print("а списки склеиваются:", [1, 2, 3] + [10, 20, 30])

сумма поэлементно: [11 22 33 44 55]
умножение поэлементно: [ 10  40  90 160 250]
возведение в степень: [ 1  4  9 16 25]
а списки склеиваются: [1, 2, 3, 10, 20, 30]


In [61]:
# Насколько быстрее — измерим
size = 1_000_000
py_list = list(range(size))
np_array = np.arange(size)

%timeit sum(x * x for x in py_list)
%timeit (np_array ** 2).sum()

55.1 ms ± 319 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)
612 µs ± 108 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)


**Вывод:** любые вычисления над числовыми данными пишем через NumPy, а не через циклы. Это называется **векторизацией**, и весь дальнейший курс на ней стоит.

In [62]:
# Матрицы: то, как в ML хранятся данные
X = np.array([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0],
    [7.0, 8.0, 9.0],
    [10.0, 11.0, 12.0],
])

print("форма (объектов, признаков):", X.shape)
print("тип данных:", X.dtype)
print("\nсреднее по каждому признаку:", X.mean(axis=0))
print("среднее по каждому объекту:  ", X.mean(axis=1))

форма (объектов, признаков): (4, 3)
тип данных: float64

среднее по каждому признаку: [5.5 6.5 7.5]
среднее по каждому объекту:   [ 2.  5.  8. 11.]


Запомните `axis`: `axis=0` — идём вдоль строк, то есть считаем **по столбцам**. `axis=1` — наоборот. Это источник половины ошибок у новичков.

In [63]:
# Broadcasting: массивы разной формы автоматически «растягиваются»
print("каждый элемент + 100:\n", X + 100)

column_means = X.mean(axis=0)
print("\nвычли среднее из каждого столбца (центрирование):\n", X - column_means)

каждый элемент + 100:
 [[101. 102. 103.]
 [104. 105. 106.]
 [107. 108. 109.]
 [110. 111. 112.]]

вычли среднее из каждого столбца (центрирование):
 [[-4.5 -4.5 -4.5]
 [-1.5 -1.5 -1.5]
 [ 1.5  1.5  1.5]
 [ 4.5  4.5  4.5]]


In [64]:
# Булева индексация — так фильтруют данные
data = np.array([12, 45, 7, 88, 23, 56, 3, 91])

mask = data > 30
print("маска:", mask)
print("значения больше 30:", data[mask])
print("сколько их:", mask.sum())
print("замена на месте:", np.where(data > 30, 0, data))

маска: [False  True False  True False  True False  True]
значения больше 30: [45 88 56 91]
сколько их: 4
замена на месте: [12  0  7  0 23  0  3  0]


In [65]:
# Форма массива меняется без копирования данных
v = np.arange(12)

print("исходный вектор:", v)
print("\nкак матрица 3x4:\n", v.reshape(3, 4))
print("\nкак столбец (12, 1):\n", v.reshape(-1, 1)[:4], "...")

исходный вектор: [ 0  1  2  3  4  5  6  7  8  9 10 11]

как матрица 3x4:
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]

как столбец (12, 1):
 [[0]
 [1]
 [2]
 [3]] ...


Метод `reshape` — один из самых частых источников путаницы. Запомните: `-1` означает «вычисли эту размерность автоматически». А `reshape(-1, 1)` превращает вектор в столбец — именно этого часто требует scikit-learn.

In [66]:
# Полезные операции, которые пригодятся в течение всего курса
scores = np.array([0.82, 0.61, 0.95, 0.73, 0.88])
names = np.array(["A", "B", "C", "D", "E"])

order = np.argsort(scores)[::-1]        # индексы по убыванию
print("порядок по убыванию:", order)
print("модели по качеству:", names[order])
print("лучшая:", names[scores.argmax()])

print("\nнакопленная сумма:", np.cumsum([1, 2, 3, 4]))
print("уникальные значения:", np.unique([1, 2, 2, 3, 3, 3]))

порядок по убыванию: [2 4 0 3 1]
модели по качеству: ['C' 'E' 'A' 'D' 'B']
лучшая: C

накопленная сумма: [ 1  3  6 10]
уникальные значения: [1 2 3]


`argsort` возвращает не значения, а **индексы** — это позволяет отсортировать один массив по значениям другого. Приём понадобится, когда будем сортировать объекты по предсказанной вероятности.

In [67]:
# Статистики по осям: частая операция при работе с матрицей объект-признак
rng = np.random.default_rng(RANDOM_STATE)
sample = rng.normal(loc=50, scale=10, size=(6, 4)).round(1)
print("данные:\n", sample)

print("\nсреднее по признакам:   ", sample.mean(axis=0).round(2))
print("отклонение по признакам:", sample.std(axis=0).round(2))
print("медиана по признакам:   ", np.median(sample, axis=0).round(2))
print("\nмин и макс по всей матрице:", sample.min(), sample.max())

данные:
 [[53.  39.6 57.5 59.4]
 [30.5 37.  51.3 46.8]
 [49.8 41.5 58.8 57.8]
 [50.7 61.3 54.7 41.4]
 [53.7 40.4 58.8 49.5]
 [48.2 43.2 62.2 48.5]]

среднее по признакам:    [47.65 43.83 57.22 50.57]
отклонение по признакам: [7.89 8.03 3.45 6.24]
медиана по признакам:    [50.25 40.95 58.15 49.  ]

мин и макс по всей матрице: 30.5 62.2


### Вопрос

Почему `sample.mean(axis=0)` даёт четыре числа, а `sample.mean(axis=1)` — шесть? Как это запомнить раз и навсегда?

_Ваш ответ:_



### Задание 1

Дан массив температур за месяц. Посчитайте и сохраните в переменные:

1. `days_above` — сколько дней температура была выше средней за месяц
2. `t_max`, `t_min` — максимальная и минимальная температура
3. `deviations` — массив отклонений каждого дня от среднего

In [68]:
rng = np.random.default_rng(RANDOM_STATE)
temperatures = rng.normal(loc=18, scale=6, size=30).round(1)
print(temperatures)

mean_tmp = temperatures.mean()
days_above = (temperatures > mean_tmp).sum()

t_max = temperatures.max()

t_min = temperatures.min()

deviations = temperatures - mean_tmp

[19.8 11.8 22.5 23.6  6.3 10.2 18.8 16.1 17.9 12.9 23.3 22.7 18.4 24.8
 20.8 12.8 20.2 12.2 23.3 17.7 16.9 13.9 25.3 17.1 15.4 15.9 21.2 20.2
 20.5 20.6]


In [69]:
# Проверка задания 1
assert int(days_above) == 16, f"дней выше среднего: {days_above}"
assert round(float(t_max), 2) == 25.30, f"максимум: {t_max}"
assert round(float(t_min), 2) == 6.30, f"минимум: {t_min}"
assert len(deviations) == 30, f"в отклонениях {len(deviations)} значений, а нужно 30"
assert round(float(deviations.sum()), 6) == 0.0, \
    f"сумма отклонений от среднего должна быть нулевой, а получилась {deviations.sum()}"
print("Задание 1 — верно ✓")

Задание 1 — верно ✓


In [70]:
# Объединение массивов — понадобится при сборке признаков
a = np.array([[1, 2], [3, 4]])
b = np.array([[5, 6], [7, 8]])

print("по вертикали (добавили объекты):\n", np.vstack([a, b]))
print("\nпо горизонтали (добавили признаки):\n", np.hstack([a, b]))

по вертикали (добавили объекты):
 [[1 2]
 [3 4]
 [5 6]
 [7 8]]

по горизонтали (добавили признаки):
 [[1 2 5 6]
 [3 4 7 8]]


### Задание 2 — со звёздочкой

Реализуйте функцию `normalize(X)`, которая приводит каждый признак к нулевому среднему и единичному отклонению. Только NumPy, без циклов.

Результат применения к `X_demo` сохраните в переменную `Z`.

In [73]:
def normalize(X):
  return (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-8)

X_demo = np.array([[2.0, 100.0], [4.0, 250.0], [6.0, 180.0], [8.0, 320.0]])
Z = normalize(X_demo)


In [74]:
# Проверка задания 2
assert Z.shape == (4, 2), f"форма результата {Z.shape}, а ожидалась (4, 2)"
assert round(float(Z.mean(axis=0)[0]), 2) == 0.0, "среднее первого признака не ноль"
assert round(float(Z.mean(axis=0)[1]), 2) == 0.0, "среднее второго признака не ноль"
assert round(float(Z.std(axis=0)[0]), 2) == 1.0, "отклонение первого признака не единица"
assert round(float(Z.std(axis=0)[1]), 2) == 1.0, "отклонение второго признака не единица"
assert round(float(Z[0, 0]), 2) == -1.34, f"Z[0, 0] = {Z[0, 0]}"

# Константный признак не должен превращаться в NaN
const = normalize(np.array([[5.0, 1.0], [5.0, 2.0], [5.0, 3.0]]))
assert not np.isnan(const).any(), \
    "у константного признака отклонение равно нулю — защититесь от деления на ноль"
print("Задание 2 — верно ✓")

Задание 2 — верно ✓


---
## 2. pandas: таблицы

NumPy хорош для однородных чисел. Реальные данные разнородны: строки, категории, даты, пропуски. Для них — pandas.

Работать будем с классическим датасетом о пассажирах «Титаника»: 891 строка, пятнадцать столбцов.

In [75]:
# Пробуем по очереди: seaborn -> локальная копия -> прямая ссылка
from pathlib import Path

def load_titanic():
    try:
        import seaborn as sns
        return sns.load_dataset("titanic")
    except Exception:
        pass
    for candidate in ("titanic.csv", "../../data/titanic.csv", "data/titanic.csv"):
        if Path(candidate).exists():
            return pd.read_csv(candidate)
    return pd.read_csv("https://raw.githubusercontent.com/mwaskom/"
                       "seaborn-data/master/titanic.csv")

df = load_titanic()
print("размер таблицы:", df.shape)
df.head()

размер таблицы: (891, 15)


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


Что в столбцах:

| столбец | смысл |
|---|---|
| `survived` | выжил (1) или нет (0) |
| `pclass` | класс каюты: 1, 2 или 3 |
| `sex`, `age` | пол и возраст |
| `sibsp`, `parch` | число супругов и братьев / родителей и детей на борту |
| `fare` | стоимость билета |
| `embarked`, `embark_town` | порт посадки: буква и название |
| `class`, `who`, `adult_male`, `alone`, `alive` | производные столбцы, дублируют другие |
| `deck` | палуба, известна далеко не для всех |

In [76]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    object  
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    object  
 8   class        891 non-null    category
 9   who          891 non-null    object  
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    object  
 13  alive        891 non-null    object  
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), object(5)
memory usage: 80.7+ KB


In [77]:
# describe() — числовые столбцы, describe(include="object") — остальные
df.describe().T

,count,mean,std,min,25%,50%,75%,max
survived,891.0,0.383838,0.486592,0.00,0.0000,0.0000,1.0,1.0000
pclass,891.0,2.308642,0.836071,1.00,2.0000,3.0000,3.0,3.0000
age,714.0,29.699118,14.526497,0.42,20.1250,28.0000,38.0,80.0000
sibsp,891.0,0.523008,1.102743,0.00,0.0000,0.0000,1.0,8.0000
parch,891.0,0.381594,0.806057,0.00,0.0000,0.0000,0.0,6.0000
fare,891.0,32.204208,49.693429,0.00,7.9104,14.4542,31.0,512.3292


### Series и DataFrame

В pandas два объекта. **DataFrame** — вся таблица. **Series** — один столбец: массив значений плюс индекс.

Почти всё, что умеет NumPy, Series умеет тоже, но с двумя добавками: у значений есть метки (индекс) и пропуски не ломают вычисления.

In [78]:
ages = df["age"]        # это Series

print("тип объекта:", type(ages).__name__)
print("длина:", len(ages))
print("индекс:", ages.index[:5].tolist(), "...")
print("\nсреднее (пропуски пропускаются автоматически):", round(ages.mean(), 2))
print("непустых значений:", ages.count(), "из", len(ages))

# Один столбец — Series, список столбцов — снова DataFrame
print("\nтип df[['age']]:", type(df[["age"]]).__name__)

тип объекта: Series
длина: 891
индекс: [0, 1, 2, 3, 4] ...

среднее (пропуски пропускаются автоматически): 29.7
непустых значений: 714 из 891

тип df[['age']]: DataFrame


### Выбор данных: `loc` и `iloc`

Главное различие в pandas, которое нужно усвоить сразу:

- **`iloc`** — выбор по **позициям**, как в списках Python. Правая граница среза **не включается**
- **`loc`** — выбор по **меткам** индекса и именам столбцов. Правая граница среза **включается**

Сейчас индекс таблицы — это просто номера от 0 до 890, поэтому разница кажется несущественной. Она перестанет быть такой, как только индексом станут даты или идентификаторы клиентов.

In [79]:
# iloc: строки и столбцы по номерам
print("первая строка, первые три столбца:")
print(df.iloc[0, :3])

print("\nстроки с 5 по 7 (8-я не входит), столбцы 1 и 3:")
display(df.iloc[5:8, [1, 3]])

print("\nпоследняя строка таблицы, столбец fare:", df.iloc[-1]["fare"])

первая строка, первые три столбца:
survived       0
pclass         3
sex         male
Name: 0, dtype: object

строки с 5 по 7 (8-я не входит), столбцы 1 и 3:


,pclass,age
5,3,NaN
6,1,54.0
7,3,2.0



последняя строка таблицы, столбец fare: 7.75


In [80]:
# loc: то же самое, но по именам
print("строки с меткой 5 по 8 включительно, выбранные столбцы:")
display(df.loc[5:8, ["sex", "age", "fare"]])

print("\nодно значение: возраст пассажира с меткой 3 —", df.loc[3, "age"])

строки с меткой 5 по 8 включительно, выбранные столбцы:


,sex,age,fare
5,male,NaN,8.4583
6,male,54.0,51.8625
7,male,2.0,21.0750
8,female,27.0,11.1333



одно значение: возраст пассажира с меткой 3 — 35.0


**Ловушка.** `df.iloc[5:8]` вернёт три строки, а `df.loc[5:8]` — четыре. Срез по меткам включает правый конец, потому что метка не обязана быть числом: у среза `loc["Иванов":"Петров"]` понятие «следующая после Петрова» отсутствует.

In [81]:
# loc с условием — самая частая форма записи в реальном коде
display(df.loc[df["age"] > 70, ["sex", "age", "pclass", "survived"]])

# at и iat — для одного значения, работают быстрее loc и iloc
print("возраст пассажира с меткой 10:", df.at[10, "age"])
print("значение в позиции (10, 3):    ", df.iat[10, 3])

,sex,age,pclass,survived
96,male,71.0,1,0
116,male,70.5,3,0
493,male,71.0,1,0
630,male,80.0,1,1
851,male,74.0,3,0


возраст пассажира с меткой 10: 4.0
значение в позиции (10, 3):     4.0


### Задание 3

Потренируйте `loc` и `iloc`. Сохраните результаты в переменные:

1. `age_10th` — возраст пассажира, стоящего в таблице десятым по счёту (то есть в позиции 9)
2. `fare_last` — стоимость билета последнего пассажира в таблице
3. `mean_age_first_100` — средний возраст среди первых ста строк
4. `n_women_first_100` — сколько женщин среди первых ста строк
5. `oldest_survivor_age` — возраст самого старшего из выживших

In [85]:
age_10th = df.iloc[9]["age"]

fare_last = df.iloc[-1]["fare"]

first_100 = df.iloc[:100]
mean_age_first_100 = first_100["age"].mean()

n_women_first_100 = (first_100["sex"] == "female").sum()

oldest_survivor_age = df.loc[df["survived"] == 1, "age"].max()

In [86]:
# Проверка задания 3
assert round(float(age_10th), 2) == 14.00, f"age_10th = {age_10th}"
assert round(float(fare_last), 2) == 7.75, f"fare_last = {fare_last}"
assert round(float(mean_age_first_100), 2) == 27.47, \
    f"mean_age_first_100 = {mean_age_first_100} (пропуски в age считать не нужно)"
assert int(n_women_first_100) == 39, f"n_women_first_100 = {n_women_first_100}"
assert round(float(oldest_survivor_age), 2) == 80.00, \
    f"oldest_survivor_age = {oldest_survivor_age}"
print("Задание 3 — верно ✓")

Задание 3 — верно ✓


### Фильтрация

Фильтр — это булева маска: Series из True и False той же длины, что и таблица. Маску кладут в `loc` или прямо в квадратные скобки.

In [87]:
mask = df["age"] > 60
print("тип маски:", type(mask).__name__, "| длина:", len(mask))
print("подходит строк:", mask.sum())

display(df.loc[mask, ["sex", "age", "pclass", "fare"]].head())

тип маски: Series | длина: 891
подходит строк: 22


,sex,age,pclass,fare
33,male,66.0,2,10.5000
54,male,65.0,1,61.9792
96,male,71.0,1,34.6542
116,male,70.5,3,7.7500
170,male,61.0,1,33.5000


In [88]:
# Несколько условий: & вместо and, | вместо or, ~ вместо not.
# Каждое условие ОБЯЗАТЕЛЬНО в скобках — приоритет операций иначе всё сломает
women_first_class = df[(df["sex"] == "female") & (df["pclass"] == 1)]
print("женщин в первом классе:", len(women_first_class))

young_or_old = df[(df["age"] < 10) | (df["age"] > 70)]
print("детей и стариков:", len(young_or_old))

not_third = df[~(df["pclass"] == 3)]
print("не в третьем классе:", len(not_third))

женщин в первом классе: 94
детей и стариков: 67
не в третьем классе: 400


**Почему не `and`.** Оператор `and` пытается привести весь столбец к одному True или False и падает с ошибкой «The truth value of a Series is ambiguous». Поэлементные версии — это `&`, `|`, `~`. И скобки: `df["age"] > 20 & df["age"] < 40` без скобок означает совсем не то, что кажется.

In [89]:
# isin — вместо длинной цепочки ==
southern_ports = df[df["embarked"].isin(["C", "Q"])]
print("село в Шербуре или Квинстауне:", len(southern_ports))

# between — вместо двух неравенств, границы включаются
middle_aged = df[df["age"].between(30, 40)]
print("в возрасте от 30 до 40 включительно:", len(middle_aged))

# query — то же самое строкой, читается легче при длинных условиях
expensive = df.query("fare > 100 and pclass == 1")
print("дорогие билеты первого класса:", len(expensive))

село в Шербуре или Квинстауне: 245
в возрасте от 30 до 40 включительно: 180
дорогие билеты первого класса: 53


In [90]:
# Пропуски в фильтре ведут себя особым образом: NaN не равен ничему,
# поэтому строки с пропущенным возрастом не попадут НИ в одну из этих выборок
print("возраст > 30:", (df["age"] > 30).sum())
print("возраст <= 30:", (df["age"] <= 30).sum())
print("сумма:", (df["age"] > 30).sum() + (df["age"] <= 30).sum(), "а строк в таблице:", len(df))
print("\nразницу дают пропуски:", df["age"].isna().sum())

возраст > 30: 305
возраст <= 30: 409
сумма: 714 а строк в таблице: 891

разницу дают пропуски: 177


### Задание 4

Отфильтруйте данные и сохраните результаты:

1. `n_target` — сколько женщин из первого или второго класса в возрасте от 20 до 40 лет включительно
2. `mean_fare_target` — их средний тариф
3. `n_child_third` — сколько детей младше 12 лет ехало третьим классом
4. `n_no_family` — сколько пассажиров ехали без родственников (`sibsp` и `parch` равны нулю)
5. `survival_no_family` — доля выживших среди них

In [91]:
target_mask = (df["pclass"].between(1, 2)) & (df["sex"] == "female") & (df["age"].between(20, 40))
n_target = target_mask.sum()

mean_fare_target = df[target_mask]["fare"].mean()

n_child_third = ((df["pclass"] == 3) & (df["age"] < 12)).sum()

no_family_mask = (df["sibsp"] == 0) & (df["parch"] == 0)
n_no_family = no_family_mask.sum()

no_family = df[no_family_mask]
survival_no_family = (no_family["survived"] == 1).sum() / no_family["survived"].count()

In [92]:
# Проверка задания 4
assert int(n_target) == 90, f"n_target = {n_target}"
assert round(float(mean_fare_target), 2) == 68.43, f"mean_fare_target = {mean_fare_target}"
assert int(n_child_third) == 47, f"n_child_third = {n_child_third}"
assert int(n_no_family) == 537, f"n_no_family = {n_no_family}"
assert round(float(survival_no_family), 2) == 0.30, \
    f"survival_no_family = {survival_no_family}"
print("Задание 4 — верно ✓")

Задание 4 — верно ✓


### Столбцы: добавить, удалить, переименовать

Правило, которое сэкономит вам часы отладки: **не меняйте исходный датафрейм**. Сделайте копию через `.copy()` и работайте с ней. Иначе после десятка ячеек вы не сможете сказать, что лежит в `df`.

In [93]:
work = df.copy()

# Добавление — просто присваивание по новому имени
work["family_size"] = work["sibsp"] + work["parch"] + 1
work["is_child"] = work["age"] < 12

# assign возвращает НОВЫЙ датафрейм и позволяет считать столбцы цепочкой
work = work.assign(
    fare_per_person=lambda d: d["fare"] / d["family_size"],
    fare_rounded=lambda d: d["fare"].round(0),
)

# insert ставит столбец на конкретную позицию
work.insert(0, "passenger_id", range(1, len(work) + 1))

display(work[["passenger_id", "family_size", "is_child",
              "fare", "fare_per_person"]].head())

,passenger_id,family_size,is_child,fare,fare_per_person
0,1,2,False,7.2500,3.62500
1,2,2,False,71.2833,35.64165
2,3,1,False,7.9250,7.92500
3,4,2,False,53.1000,26.55000
4,5,1,False,8.0500,8.05000


In [94]:
# Удаление. axis=1 и columns= — одно и то же, второе читается лучше
work = work.drop(columns=["fare_rounded"])

# Несколько сразу: уберём столбцы, дублирующие другие
duplicates = ["class", "who", "adult_male", "embark_town", "alive", "alone"]
work = work.drop(columns=duplicates)

print("столбцов осталось:", work.shape[1])
print(list(work.columns))

# pop удаляет столбец и возвращает его — удобно, когда он ещё нужен
deck = work.pop("deck")
print("\nвынули deck, в таблице теперь столбцов:", work.shape[1])

столбцов осталось: 13
['passenger_id', 'survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'deck', 'family_size', 'is_child', 'fare_per_person']

вынули deck, в таблице теперь столбцов: 12


In [95]:
# Переименование — только нужные столбцы, остальные не трогаются
work = work.rename(columns={"sibsp": "siblings_spouses",
                            "parch": "parents_children"})
print(list(work.columns))

# Удаление строк — тот же drop, но по индексу
without_first_five = work.drop(index=[0, 1, 2, 3, 4])
print("\nбыло строк:", len(work), "стало:", len(without_first_five))

['passenger_id', 'survived', 'pclass', 'sex', 'age', 'siblings_spouses', 'parents_children', 'fare', 'embarked', 'family_size', 'is_child', 'fare_per_person']

было строк: 891 стало: 886


### Задание 5

Соберите рабочую копию таблицы. Начните с `df.copy()` и сохраните её в `clean`:

1. Добавьте столбец `family_size` — число родственников на борту плюс сам пассажир
2. Добавьте `fare_per_person` — тариф, делённый на размер семьи
3. Добавьте `age_group` — категория: `"ребёнок"` для возраста меньше 12, `"взрослый"` для остальных (пропуски пусть станут взрослыми)
4. Удалите столбцы `class`, `who`, `adult_male`, `embark_town`, `alive`, `alone`, `deck`
5. Сохраните в `n_cols` число оставшихся столбцов, в `mean_fare_per_person` — средний тариф на человека, в `n_children` — число детей

In [96]:
clean = df.copy()

clean["family_size"] = clean["sibsp"] + clean["parch"] + 1

clean["fare_per_person"] = clean["fare"] / clean["family_size"]

clean["age_group"] = np.where(df["age"] < 12, "ребёнок", "взрослый")

clean = clean.drop(columns=["class", "who", "adult_male", "embark_town", "alive", "alone", "deck"])

n_cols = clean.shape[1]
mean_fare_per_person = clean["fare_per_person"].mean()

n_children = (clean["age_group"] == "ребёнок").sum()

In [97]:
# Проверка задания 5
assert "family_size" in clean.columns, "нет столбца family_size"
assert "fare_per_person" in clean.columns, "нет столбца fare_per_person"
assert "age_group" in clean.columns, "нет столбца age_group"
assert "deck" not in clean.columns, "столбец deck нужно было удалить"
assert "alive" not in clean.columns, "столбец alive нужно было удалить"
assert int(n_cols) == 11, f"столбцов {n_cols}, а должно остаться 11"
assert int(clean["family_size"].max()) == 11, \
    f"максимальный размер семьи {clean['family_size'].max()}, ожидается 11"
assert round(float(mean_fare_per_person), 2) == 19.92, \
    f"mean_fare_per_person = {mean_fare_per_person}"
assert int(n_children) == 68, f"n_children = {n_children}"
assert len(clean) == 891, "строки удалять не требовалось"
print("Задание 5 — верно ✓")

Задание 5 — верно ✓


### Сортировка и порядок

In [98]:
# sort_values — по одному или нескольким столбцам
top_fares = clean.sort_values("fare", ascending=False).head(5)
display(top_fares[["sex", "pclass", "fare", "family_size", "survived"]])

# nlargest делает то же самое короче и быстрее
print("три самых дорогих билета:", clean.nlargest(3, "fare")["fare"].tolist())

# Сортировка по двум ключам сразу
by_class_age = clean.sort_values(["pclass", "age"], ascending=[True, False])
print("\nстарейший пассажир первого класса:",
      by_class_age.iloc[0]["age"])

,sex,pclass,fare,family_size,survived
679,male,1,512.3292,2,1
258,female,1,512.3292,1,1
737,male,1,512.3292,1,1
88,female,1,263.0000,6,1
438,male,1,263.0000,6,0


три самых дорогих билета: [512.3292, 512.3292, 512.3292]

старейший пассажир первого класса: 80.0


In [99]:
# После сортировки индекс перемешан — reset_index наводит порядок
mixed = clean.sort_values("fare", ascending=False).head(3)
print("индекс после сортировки:", mixed.index.tolist())
print("после reset_index:      ",
      mixed.reset_index(drop=True).index.tolist())

индекс после сортировки: [679, 258, 737]
после reset_index:       [0, 1, 2]


### Пропуски: механика

Сегодня — только техника: как найти и как заполнить. Чем именно заполнять и как это влияет на модель — тема следующего занятия.

In [100]:
missing = pd.DataFrame({
    "пропусков": df.isna().sum(),
    "доля, %": (df.isna().mean() * 100).round(1),
})
display(missing[missing["пропусков"] > 0].sort_values("пропусков", ascending=False))

,пропусков,"доля, %"
deck,688,77.2
age,177,19.9
embarked,2,0.2
embark_town,2,0.2


In [101]:
# dropna убирает строки, fillna заполняет
print("строк всего:                 ", len(df))
print("после dropna по всей таблице:", len(df.dropna()))
print("после dropna только по age:  ", len(df.dropna(subset=["age"])))

filled = df["age"].fillna(df["age"].median())
print("\nпропусков в age было:", df["age"].isna().sum(),
      "| стало:", filled.isna().sum())
print("медиана, которой заполняли:", df["age"].median())

строк всего:                  891
после dropna по всей таблице: 182
после dropna только по age:   714

пропусков в age было: 177 | стало: 0
медиана, которой заполняли: 28.0


`dropna()` без аргументов выбросил почти всю таблицу — из-за столбца `deck`, где пропусков 688 из 891. Это типичная ошибка: одна строка кода, и от данных осталась пятая часть.

### Типы данных

In [102]:
print(df.dtypes.to_string())

# astype меняет тип. Для категорий это ещё и экономия памяти
before = df["sex"].memory_usage(deep=True)
after = df["sex"].astype("category").memory_usage(deep=True)
print(f"\nстолбец sex: {before} байт -> {after} байт после astype('category')")

# Приведение к целому не сработает, пока в столбце есть пропуски
print("\nage с пропусками привести к int нельзя, а после заполнения — можно:")
print(df["age"].fillna(df["age"].median()).astype(int).head().tolist())

survived          int64
pclass            int64
sex              object
age             float64
sibsp             int64
parch             int64
fare            float64
embarked         object
class          category
who              object
adult_male         bool
deck           category
embark_town      object
alive            object
alone              bool

столбец sex: 47983 байт -> 1239 байт после astype('category')

age с пропусками привести к int нельзя, а после заполнения — можно:
[22, 38, 26, 35, 35]


### `apply` и `map`

In [103]:
demo = df.copy()

# map — по словарю или простой функции, только для Series
demo["sex_code"] = demo["sex"].map({"male": 0, "female": 1})

# apply по столбцу
demo["fare_category"] = demo["fare"].apply(
    lambda x: "дёшево" if x < 10 else ("средне" if x < 50 else "дорого")
)

# apply по строкам целиком: axis=1
demo["family_size"] = demo.apply(
    lambda row: row["sibsp"] + row["parch"] + 1, axis=1
)

display(demo[["sex", "sex_code", "fare", "fare_category", "family_size"]].head())

,sex,sex_code,fare,fare_category,family_size
0,male,0,7.2500,дёшево,2
1,female,1,71.2833,дорого,2
2,female,1,7.9250,дёшево,1
3,female,1,53.1000,дорого,2
4,male,0,8.0500,дёшево,1


In [107]:
# Сравним скорость: apply по строкам против векторизованной записи
import time

start = time.perf_counter()
_ = df.apply(lambda r: r["sibsp"] + r["parch"] + 1, axis=1)
t_apply = time.perf_counter() - start

start = time.perf_counter()
_ = df["sibsp"] + df["parch"] + 1
t_vector = time.perf_counter() - start

print(f"apply(axis=1):   {t_apply * 1000:.2f} мс")
print(f"векторизованно:  {t_vector * 1000:.2f} мс")
print(f"разница в {t_apply / t_vector:.0f} раз")

apply(axis=1):   6.68 мс
векторизованно:  0.52 мс
разница в 13 раз


### Вопрос

На выборке из 900 строк разница во времени незаметна для человека. Почему тогда вообще стоит обращать на это внимание?

_Ваш ответ:_



In [108]:
# Работа со строками — доступ через .str
names = pd.Series(["Иванов Иван", "Петрова Мария", "Сидоров Пётр"])

print("в верхнем регистре:", names.str.upper().tolist())
print("длина:", names.str.len().tolist())
print("фамилии:", names.str.split().str[0].tolist())
print("содержит 'ов':", names.str.contains("ов").tolist())

в верхнем регистре: ['ИВАНОВ ИВАН', 'ПЕТРОВА МАРИЯ', 'СИДОРОВ ПЁТР']
длина: [11, 13, 12]
фамилии: ['Иванов', 'Петрова', 'Сидоров']
содержит 'ов': [True, True, True]


In [109]:
# Даты: отдельный тип со своими операциями
dates = pd.Series(pd.to_datetime([
    "2020-03-15", "2020-07-04", "2020-12-31", "2021-01-01"
]))

info = pd.DataFrame({
    "дата": dates,
    "год": dates.dt.year,
    "месяц": dates.dt.month,
    "день_недели": dates.dt.dayofweek,
    "выходной": dates.dt.dayofweek >= 5,
})
display(info)

print("\nразница в днях между первой и последней:",
      (dates.iloc[-1] - dates.iloc[0]).days)

,дата,год,месяц,день_недели,выходной
0,2020-03-15,2020,3,6,True
1,2020-07-04,2020,7,5,True
2,2020-12-31,2020,12,3,False
3,2021-01-01,2021,1,4,False



разница в днях между первой и последней: 292


### Группировка

`groupby` разбивает таблицу на части по значению столбца и считает что-нибудь внутри каждой. Сегодня это для нас инструмент; искать в таких таблицах закономерности будем на следующем занятии.

In [110]:
# value_counts — самый быстрый способ посмотреть на категорию
print(df["pclass"].value_counts())
print()
print(df["embarked"].value_counts(normalize=True).round(3))

pclass
3    491
1    216
2    184
Name: count, dtype: int64

embarked
S    0.724
C    0.189
Q    0.087
Name: proportion, dtype: float64


In [111]:
# groupby + agg: несколько статистик сразу, с понятными именами
summary = df.groupby("pclass").agg(
    средний_возраст=("age", "mean"),
    средний_тариф=("fare", "mean"),
    пассажиров=("survived", "size"),
).round(2)
display(summary)

# Группировка по двум столбцам даёт составной индекс
two_keys = df.groupby(["sex", "pclass"])["fare"].mean().round(2)
print(two_keys)

,средний_возраст,средний_тариф,пассажиров
pclass,,,
1,38.23,84.15,216
2,29.88,20.66,184
3,25.14,13.68,491


sex     pclass
female  1         106.13
        2          21.97
        3          16.12
male    1          67.23
        2          19.74
        3          12.66
Name: fare, dtype: float64


In [112]:
# pivot_table разворачивает вторую группировку в столбцы
display(df.pivot_table(values="fare", index="sex",
                       columns="pclass", aggfunc="mean").round(2))

# transform возвращает результат в форме исходной таблицы —
# удобно, когда нужно сравнить объект со своей группой
mean_by_class = df.groupby("pclass")["fare"].transform("mean")
print("\nпервые пять тарифов:      ", df["fare"].head().round(2).tolist())
print("средний тариф их класса:  ", mean_by_class.head().round(2).tolist())

pclass,1,2,3
sex,,,
female,106.13,21.97,16.12
male,67.23,19.74,12.66



первые пять тарифов:       [7.25, 71.28, 7.92, 53.1, 8.05]
средний тариф их класса:   [13.68, 84.15, 13.68, 84.15, 13.68]


### Задание 6

Поработайте с группировками. Сохраните результаты:

1. `mean_age_by_class` — Series со средним возрастом по каждому классу каюты
2. `age_class_3` — средний возраст в третьем классе
3. `richest_port` — буква порта (`embarked`) с самым высоким средним тарифом
4. `biggest_group_size` — размер самой многочисленной группы «пол + класс»
5. `top3_fare_sum` — сумма трёх самых дорогих билетов

In [129]:
mean_age_by_class = df.groupby("pclass")["age"].mean()

age_class_3 = mean_age_by_class.loc[3]

richest_port = df.groupby("embarked")["fare"].mean().idxmax()

biggest_group_size = df.groupby(["sex", "pclass"]).size().max()

top3_fare_sum = df["fare"].sort_values(ascending=False).head(3).sum()

In [130]:
# Проверка задания 6
assert len(mean_age_by_class) == 3, "в mean_age_by_class должно быть три группы"
assert round(float(age_class_3), 2) == 25.14, f"age_class_3 = {age_class_3}"
assert richest_port == "C", f"richest_port = {richest_port}"
assert int(biggest_group_size) == 347, f"biggest_group_size = {biggest_group_size}"
assert round(float(top3_fare_sum), 2) == 1536.99, f"top3_fare_sum = {top3_fare_sum}"
print("Задание 6 — верно ✓")

Задание 6 — верно ✓


---
## 3. Точка отсчёта

Инструменты есть — теперь про то, ради чего они нужны. Задача: предсказать, выжил ли пассажир.

Порядок действий на первом занятии важнее любой модели:

1. Сначала **делим данные** — до любой обработки
2. Строим **тупейший baseline** — точку отсчёта
3. И только потом имеет смысл обучать что-то осмысленное

Третий шаг — тема следующих занятий. Первые два разберём сейчас.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

features = ["pclass", "sibsp", "parch", "fare"]
X = df[features]
y = df["survived"]

# stratify сохраняет пропорцию классов в обеих частях
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"обучающая выборка: {X_train.shape[0]} объектов")
print(f"тестовая выборка:  {X_test.shape[0]} объектов")
print(f"доля выживших в обучающей: {y_train.mean():.3f}")
print(f"доля выживших в тестовой:  {y_test.mean():.3f}")

**Почему делим до всего остального.** Любое действие, подсмотренное на тестовой части — заполнение пропусков её медианой, отбор признаков по всей таблице, — завышает оценку качества. Это называется утечкой, и на следующем занятии мы увидим её в эксперименте.

In [ ]:
# Baseline: предсказываем самый частый класс, ни на что не глядя
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
dummy_acc = accuracy_score(y_test, dummy.predict(X_test))

print(f"«все погибли»:     accuracy = {dummy_acc:.3f}")
print(f"доля погибших в тесте:        {1 - y_test.mean():.3f}")
print("\nЭто число — точка отсчёта. Модель, которая его не побьёт, бесполезна.")

### Вопрос

Baseline «все погибли» даёт около 0.61 точности, ничего не зная о пассажирах. Представьте задачу, где событие происходит в одном случае из тысячи. Какую точность покажет такой baseline и что это говорит о самой метрике?

_Ваш ответ:_



### Задание 7

Сравните три тривиальные стратегии и сохраните их точность:

1. `acc_most_frequent` — всегда самый частый класс
2. `acc_stratified` — случайный ответ с сохранением пропорции классов (`strategy="stratified"`, `random_state=RANDOM_STATE`)
3. `acc_constant_1` — всегда отвечать «выжил» (`strategy="constant"`, `constant=1`)

И ответьте себе: какая из них честнее всего показывает, насколько задача трудная?

In [ ]:
# ВАШ КОД ЗДЕСЬ


In [ ]:
# Проверка задания 7
assert round(float(acc_most_frequent), 2) == 0.61, \
    f"acc_most_frequent = {acc_most_frequent}"
assert round(float(acc_stratified), 2) == 0.52, \
    f"acc_stratified = {acc_stratified} (не забудьте random_state=RANDOM_STATE)"
assert round(float(acc_constant_1), 2) == 0.39, \
    f"acc_constant_1 = {acc_constant_1}"
print("Задание 7 — верно ✓")

### Столбец, которого не должно быть

Напоследок — короткая демонстрация того, о чём будет много разговоров весь курс. В исходной таблице есть столбец `alive`.

In [ ]:
print(pd.crosstab(df["survived"], df["alive"]))

# Правило в одну строку, без всякого обучения
rule = (df["alive"] == "yes").astype(int)
print(f"\nточность правила «alive == yes»: {accuracy_score(df['survived'], rule):.3f}")

### Вопрос

Правило на одном столбце дало стопроцентную точность. Это хорошая новость или плохая? Как вообще замечать такие столбцы в своих данных?

_Ваш ответ:_



---
## Что дальше

На следующем занятии:
- разведочный анализ: графики, распределения, связи между признаками
- пропуски и выбросы: не только как найти, но и что с ними делать
- конструирование признаков и сборка пайплайна
- утечка данных в живом эксперименте

**Домашнее задание 1** выдаётся после лекции 2.

### Полезное

- [pandas — 10 minutes to pandas](https://pandas.pydata.org/docs/user_guide/10min.html)
- [Шпаргалка по pandas](https://pandas.pydata.org/Pandas_Cheat_Sheet.pdf)
- [NumPy — the absolute basics for beginners](https://numpy.org/doc/stable/user/absolute_beginners.html)